# Index · Unit 6 — Structured I/O & reliability

Same shape as Units 1–5. For every component in **Section 7 of `index.md`**, two cells:

- **From scratch** — the mechanism by hand, no framework. Runs on plain Python /
  scripted *fake model* (no API key needed).
- **SOTA** — how it's actually done today (constrained decoding, retry libs, saga/idempotency).
  Some need `OPENAI_API_KEY`.

Each component lists **📚 Resources** (also consolidated in `RESOURCES.md`).

Components: **1** structured outputs · **2** error handling (retries / fallbacks /
timeouts) · **3** idempotency & rollback.

**Intuition for the whole unit:** an agent is a *program* that happens to call a
model, and programs must be reliable. That means the model's output has to be
**machine-readable** (structured outputs), the calls around it have to **survive
failure** (retries/fallbacks/timeouts), and the actions it takes have to be **safe
to re-run** (idempotency) and **undoable** (rollback). Same pattern as before:
thin wrappers around the model, which SOTA tools package.

## Setup

In [ ]:
import os, re, json

MODEL = "gpt-4o-mini"

print("✅ key set — SOTA cells will run" if os.environ.get("OPENAI_API_KEY")
      else "⚠️  no OPENAI_API_KEY — from-scratch cells still run; SOTA cells need a key")

def chat(messages, model=MODEL, **kwargs):
    from openai import OpenAI
    return OpenAI().chat.completions.create(
        model=model, messages=messages, **kwargs).choices[0].message

## 1. Structured outputs

Free text is unreliable to parse. **Structured outputs** constrain the model to a
**schema** so the result is valid JSON you can feed straight into code — no regex,
no "sometimes it adds a preamble". From scratch you *ask* for JSON and validate;
SOTA uses **constrained decoding** so the tokens *can't* violate the schema.

**From scratch:** prompt for JSON, parse, validate against a hand-written schema
check, retry once on failure.

> 📚 **Resources:** [OpenAI Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs) ·
> [Pydantic](https://docs.pydantic.dev/) · [Instructor](https://github.com/567-labs/instructor) ·
> [Outlines](https://github.com/dottxt-ai/outlines)

In [ ]:
# From scratch — parse + validate JSON against a tiny schema checker.
SCHEMA = {"name": str, "age": int, "email": str}

def validate(obj, schema):
    for key, typ in schema.items():
        if key not in obj:            return False, f"missing '{key}'"
        if not isinstance(obj[key], typ): return False, f"'{key}' must be {typ.__name__}"
    return True, "ok"

def fake_model(prompt, attempt):
    # 1st reply is malformed (has a preamble); 2nd is clean JSON.
    if attempt == 0:
        return 'Sure! Here you go: {"name": "Mia", "age": "thirty"}'
    return '{"name": "Mia", "age": 30, "email": "mia@x.com"}'

def structured(prompt, schema, max_tries=2):
    for attempt in range(max_tries):
        raw = fake_model(prompt, attempt)
        m = re.search(r"\{.*\}", raw, re.S)                 # strip any preamble
        try:
            obj = json.loads(m.group()) if m else {}
            ok, err = validate(obj, schema)
            if ok: return obj
            print(f"  attempt {attempt+1} invalid: {err}")
        except json.JSONDecodeError as e:
            print(f"  attempt {attempt+1} not JSON: {e}")
    raise ValueError("never produced valid structured output")

print(structured("Extract the person.", SCHEMA))

In [ ]:
# SOTA — strict json_schema: the decoder CANNOT emit tokens that break the schema.
def sota_structured(text):
    schema = {"type": "object", "properties": {
        "name":  {"type": "string"},
        "age":   {"type": "integer"},
        "email": {"type": "string"}},
        "required": ["name", "age", "email"], "additionalProperties": False}
    m = chat([{"role": "user", "content": f"Extract the person from: {text}"}],
             response_format={"type": "json_schema",
                              "json_schema": {"name": "person", "schema": schema, "strict": True}})
    return json.loads(m.content)      # guaranteed parseable, guaranteed to match schema

print(sota_structured("Mia is 30, reach her at mia@x.com"))

## 2. Error handling — retries, fallbacks, timeouts

Everything external fails eventually: the API 500s, times out, or rate-limits.
Reliable agents wrap calls with **retry + exponential backoff** (ride out
transient failures), a **timeout** (don't hang forever), and a **fallback** (a
cheaper model, a cached answer, or a graceful "I couldn't do that").

**From scratch:** a retry-with-backoff wrapper and a fallback wrapper over a flaky
function.

> 📚 **Resources:** [tenacity (retrying lib)](https://tenacity.readthedocs.io/) ·
> [OpenAI SDK retries & timeouts](https://github.com/openai/openai-python#retries) ·
> [AWS — exponential backoff & jitter](https://aws.amazon.com/builders-library/timeouts-retries-and-backoff-with-jitter/)

In [ ]:
# From scratch — retry w/ exponential backoff, plus a fallback.
def make_flaky(fails_then_ok):
    state = {"n": 0}
    def flaky():
        state["n"] += 1
        if state["n"] <= fails_then_ok:
            raise ConnectionError(f"transient error (call {state['n']})")
        return f"success on call {state['n']}"
    return flaky

def with_retry(fn, retries=4, base=0.1):
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            delay = base * (2 ** attempt)                   # 0.1, 0.2, 0.4, ...
            print(f"  attempt {attempt+1} failed: {e}  -> backoff {delay:.2f}s")
    raise RuntimeError("all retries exhausted")

def with_fallback(primary, fallback):
    try:    return primary()
    except Exception as e:  print(f"  primary failed ({e}); using fallback"); return fallback()

print("retry  ->", with_retry(make_flaky(2)))                       # fails twice, then ok
print("fallbk ->", with_fallback(make_flaky(99), lambda: "cached answer"))

In [ ]:
# SOTA — SDK-level timeout/retries + a model fallback (big -> small on failure).
def robust_chat(prompt):
    try:
        # the SDK already retries transient errors; add a timeout so we never hang.
        return chat([{"role": "user", "content": prompt}], timeout=10).content
    except Exception as e:
        print(f"  primary model failed ({e}); falling back to gpt-4o-mini")
        return chat([{"role": "user", "content": prompt}], model="gpt-4o-mini").content

print(robust_chat("Say 'reliable' in one word."))

## 3. Idempotency & rollback

Because we retry, an action may run **more than once** — paying twice, sending two
emails. **Idempotency** makes a repeated call a no-op: tag each action with a key;
if you've seen the key, return the stored result instead of acting again. And when
a multi-step action fails partway, **rollback** undoes the completed steps with
**compensating actions** (the saga pattern).

**From scratch:** an idempotency-key cache, and a transfer that rolls back on
failure.

> 📚 **Resources:** [Stripe — Idempotent requests](https://docs.stripe.com/api/idempotent_requests) ·
> [microservices.io — Saga pattern](https://microservices.io/patterns/data/saga.html)

In [ ]:
# From scratch — idempotency keys: the same key never acts twice.
_processed = {}
def charge(idempotency_key, amount):
    if idempotency_key in _processed:
        print(f"  duplicate '{idempotency_key}' -> returning stored result")
        return _processed[idempotency_key]
    result = {"charged": amount, "id": f"txn_{len(_processed)+1}"}
    _processed[idempotency_key] = result
    print(f"  charged ${amount}")
    return result

print(charge("order-42", 100))
print(charge("order-42", 100))     # retry with same key -> no double charge

In [ ]:
# From scratch — rollback via compensating actions (saga).
def transfer(accounts, src, dst, amt):
    done = []                                   # journal of completed steps
    try:
        accounts[src] -= amt; done.append(("refund", src, amt))
        if accounts[src] < 0: raise ValueError("insufficient funds")
        accounts[dst] += amt; done.append(("reclaim", dst, amt))
        print("  transfer committed")
    except Exception as e:
        for action, acct, a in reversed(done):  # undo in reverse order
            accounts[acct] += a if action == "refund" else -a
        print(f"  ✋ rolled back ({e})")
    return accounts

print("ok  :", transfer({"A": 100, "B": 0}, "A", "B", 60))
print("fail:", transfer({"A": 50,  "B": 0}, "A", "B", 999))   # rolls back cleanly

## Recap

Reliability = treating the agent as a real program:

| component | mechanism (from scratch) | SOTA packaging |
|---|---|---|
| structured outputs | ask for JSON, parse, validate, retry | strict `json_schema` constrained decoding, Pydantic/Instructor |
| error handling | retry+backoff, fallback wrappers | SDK retries/timeouts, tenacity, model fallback |
| idempotency & rollback | key cache + compensating-action journal | idempotency keys (Stripe), saga/transactions |

These three compose: structured outputs make failures *detectable*, retries make
them *survivable*, and idempotency+rollback make retries *safe*. The through-line
from Units 1–5 holds — each is a few lines of plain Python; SOTA swaps in a
battle-tested library or API feature.

Next in `index.md`: **Unit 7 — Evaluation & observability** (Section 8).